
# Overlay WCSim AmBe events into DAQ windows

This notebook builds synthetic acquisition windows of duration `deltat` by overlaying single-AmBe events from ROOT WCSim output. 

The digitalized hits information is used.

## Model used here

For each recorded window:

1. define a continuous source stream with activity `A` and recorded live window length `deltat_seconds = deltat_ns * 1e-9`
2. draw the number of new source decays in the corresponding window from a Poisson law:
   - if `dead_time_ns = 0`:  
     `n ~ Poisson(A * deltat_seconds)`
   - if `dead_time_ns > 0`:  
     `n ~ Poisson(A * (deltat_ns + dead_time_ns) * 1e-9)`  
     because decays during the dead gap before the window are also generated
3. sample `n` events from the input ROOT file (without replacement)
4. for each sampled event:
   - draw one absolute prompt time uniformly in the generation interval
   - compute one global time shift that places that event in the continuous stream
   - keep the event as **active** until its latest relevant hit time has passed
5. for the current recorded window:
   - collect contributions from all active events
   - keep only hits and captures whose shifted absolute times fall inside the current recorded window
   - store times relative to the start of that window
6. merge and sort the surviving hits in time

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import uproot
import matplotlib.pyplot as plt

In [ ]:
# ---------------------------
# Parameters
# ---------------------------

input_root = "../simulation/output/wcte_ambe_000_hits.root"
tree_name = "TDigiHits"

# Source activity in Bq = decays / second
factor = 5.95e-5 # scale factor to account for the amount of decays that produce neutrons
A = 3.7e6 * factor 

# Acquisition window length in ns and s
deltat_ns = 270_000.0   # 270 microseconds
deltat_s = deltat_ns * 1e-9

# Number of synthetic acquisition windows to generate
N_windows = 108_025

# Dead time between windows in ns
dead_time_ns = 7_750.0   # 7.75 microseconds

# RNG seed for reproducibility
seed = 12345
rng = np.random.default_rng(seed)

print(f"A = {A:.3f} neutrons/s")
print(f"deltat = {deltat_ns:.1f} ns = {deltat_s:.6e} s")
print(f"Expected neutrons per window = A * deltat = {A * deltat_s:.6f}")
print(f"Dead time between windows = {dead_time_ns:.1f} ns = {dead_time_ns * 1e-9:.6e} s")

In [ ]:

def poisson_n(activity_bq, deltat_seconds, rng):
    """Draw the number of source decays in one acquisition window."""
    lam = activity_bq * deltat_seconds
    return rng.poisson(lam)

# Quick check
[poisson_n(A, deltat_s, rng) for _ in range(10)]


In [ ]:
# ---------------------------
# Load the input event library
# ---------------------------

path = Path(input_root)
if not path.exists():
    raise FileNotFoundError(f"Input file not found: {path.resolve()}")

branches = [
    "event", "nsubevents", "subevent", "ndigi", "ncaptures", "sumQ",
    "has_prompt", "prompt_track_id", "prompt_track_pdg",
    "prompt_track_energy", "prompt_track_time",
    "prompt_track_creator_process",
    "tubeid", "mPMTid", "mPMT_pmtid",
    "q", "t", "x", "y", "z", "dx", "dy", "dz",
    "ntruth",
    "hit_from_prompt", "hit_from_capture", "hit_from_otherprocess",
    "capture_t", "capture_x", "capture_y", "capture_z",
    "capture_nucleus", "capture_ngamma", "capture_total_gammaE",
]

with uproot.open(path) as f:
    tree = f[tree_name]
    arr = tree.arrays(branches, library="np")

n_library_events = len(arr["event"])
print(f"Loaded {n_library_events} library events from {input_root}:{tree_name}")


## QE and SPE corrections

### Transformation from WCSim to WCTE IDs

These WCTE_pmt_id values are the ones used to identify PMT type and site through `other_mpmt_info.dict`.

In [ ]:
# ============================================================
# Convert WCSim identifiers inside each library event
# to WCTE identifiers and add them back to the arr dictionary.
#
# New fields added to arr:
#
#   arr["WCTE_slot"]    : WCTE mPMT slot ID
#   arr["WCTE_pos"]     : WCTE PMT position inside mPMT, 0-based, 0..18
#   arr["WCTE_pmt_id"]  : flattened WCTE PMT ID = slot * 19 + pos
#
# The correspondence is the following:
#
#   WCSim mPMTid  = WCTE slot 
#   WCSim mPMT_pmtid = WCTE pos + 1 (because it is 1-based in WCSim and 0-based in WCTE)
#
# There is no need of going through the tube_id.
# ============================================================


# ------------------------------------------------------------
# Add WCTE arrays event-by-event
# ------------------------------------------------------------
n_library_events = len(arr["event"])

arr["WCTE_slot"] = np.empty(n_library_events, dtype=object)
arr["WCTE_pos"] = np.empty(n_library_events, dtype=object)
arr["WCTE_pmt_id"] = np.empty(n_library_events, dtype=object)


for i in range(n_library_events):

    WCTE_slot = np.asarray(arr["mPMTid"][i], dtype=np.int32)
    WCTE_pos = np.asarray(arr["mPMT_pmtid"][i], dtype=np.int32) - 1

    if WCTE_slot.size == 0:
        arr["WCTE_slot"][i] = np.array([], dtype=np.int32)
        arr["WCTE_pos"][i] = np.array([], dtype=np.int32)
        arr["WCTE_pmt_id"][i] = np.array([], dtype=np.int32)
        continue

    if np.any((WCTE_pos < 0) | (WCTE_pos >= 19)):
        raise ValueError(f"Invalid WCTE_pos in library event {i}: range {WCTE_pos.min()}..{WCTE_pos.max()}")
    
    # Add WCTE identifiers to arr
    arr["WCTE_slot"][i] = WCTE_slot
    arr["WCTE_pos"][i] = WCTE_pos
    arr["WCTE_pmt_id"][i] = (WCTE_slot * 19 + WCTE_pos).astype(np.int32)


print(f"Added WCTE IDs to {n_library_events} library events.")

### Implementing QE correction: 0.5 factor to the WUT Ex-situ PMTs

In [ ]:
# Code to map PMTs to their type and site, to apply different constant QE factors.
# To-do in the future: apply per-PMT QE factors instead of constant ones.

import pickle

N_PMTS = 2014 #1843

mpmt_info_path = "other_mpmt_info.dict"

pmt_qe_factor = np.full(N_PMTS, 1.0, dtype=np.float32)

try:
    with open(mpmt_info_path, "rb") as f:
        mpmt_info = pickle.load(f)

    for pmt_id in range(N_PMTS):
        slot = pmt_id // 19
        if slot not in mpmt_info:
            continue

        mtype = mpmt_info[slot].get("mpmt_type", "")
        msite = mpmt_info[slot].get("mpmt_site", "")

        if msite == "TRI" and mtype == "In-situ":
            pmt_qe_factor[pmt_id] = 1.0
        elif msite == "TRI" and mtype == "Ex-situ":
            pmt_qe_factor[pmt_id] = 1.0
        elif msite == "WUT" and mtype == "In-situ":
            pmt_qe_factor[pmt_id] = 1.0
        elif msite == "WUT" and mtype == "Ex-situ":
            pmt_qe_factor[pmt_id] = 0.5

except FileNotFoundError:
    print(f"{mpmt_info_path} not found.")

except Exception as e:
    print(f"Could not load {mpmt_info_path}: {e}.")


low_qe_pmts = np.where(pmt_qe_factor == 0.5)[0]

print(f"Number of PMTs with QE factor 0.5: {len(low_qe_pmts)}")
print("First few low-QE PMT IDs:", low_qe_pmts[:20])

In [ ]:
# CAUTION: this must be run only once, because it modifies arr in-place.
#
# After this cell, the event library itself is QE-corrected, so the windows
# built from arr will automatically use the corrected hit content.

# ============================================================
# Apply QE by random hit removal at the library-event level
# ============================================================
#
# Interpretation:
#
#   pmt_qe_factor[pmt_id] = 1.0  -> keep all hits on that PMT
#   pmt_qe_factor[pmt_id] = 0.5  -> randomly keep 50% of hits on that PMT
#
# This does not rescale charge.
# It removes hit entries from all hit-level arrays consistently.
#
# Each event must already contain arr["WCTE_pmt_id"] from the previous
# WCSim -> WCTE conversion cell.
#
# Capture-level truth arrays are NOT modified here:
#
#   capture_t, capture_x, capture_y, ...
#
# because those are event/capture truth quantities, not hit-level quantities.
# A capture may remain in truth even if some/all of its PMT hits are lost
# after QE.
#
# ============================================================

# Random generator for reproducibility.
rng_qe = np.random.default_rng(12345)


# These are the arrays with one entry per digi-hit in each library event.
# Any hit removed by QE must be removed from all of these fields.
hit_level_fields = [
    "t",
    "q",
    "tubeid",
    "mPMTid",
    "mPMT_pmtid",
    "x",
    "y",
    "z",
    "dx",
    "dy",
    "dz",
    "hit_from_prompt",
    "hit_from_capture",
    "hit_from_otherprocess",
    "ntruth",

    # WCTE IDs added by the previous conversion cell
    "WCTE_slot",
    "WCTE_pos",
    "WCTE_pmt_id",
]


n_library_events = len(arr["event"])

n_hits_before_total = 0
n_hits_after_total = 0
n_removed_total = 0
n_qe05_hits_total = 0

# Optional bookkeeping arrays at event level
arr["ndigi_before_QE"] = np.zeros(n_library_events, dtype=np.int32)
arr["ndigi_removed_by_QE"] = np.zeros(n_library_events, dtype=np.int32)


for i in range(n_library_events):

    if "WCTE_pmt_id" not in arr:
        raise KeyError(
            "arr is missing 'WCTE_pmt_id'. "
            "Run the WCSim -> WCTE conversion cell before applying QE."
        )

    pmt_id = np.asarray(arr["WCTE_pmt_id"][i], dtype=np.int64)

    n_hits_before = len(pmt_id)
    n_hits_before_total += n_hits_before
    arr["ndigi_before_QE"][i] = int(n_hits_before)

    # Empty event
    if n_hits_before == 0:
        arr["ndigi"][i] = 0
        arr["sumQ"][i] = 0.0
        continue

    # --------------------------------------------------------
    # Per-hit QE probability
    # --------------------------------------------------------
    # For each hit, get the QE factor of the PMT that detected it.
    #
    # Example:
    #   if pmt_qe_factor[pmt_id] = 0.5,
    #   the hit is kept with probability 0.5.
    # --------------------------------------------------------
    keep_probability = pmt_qe_factor[pmt_id]

    # Count hits on reduced-QE PMTs, useful for diagnostics
    n_qe05_hits_total += int(np.count_nonzero(keep_probability == 0.5))

    # --------------------------------------------------------
    # Random keep/drop decision for every hit
    # --------------------------------------------------------
    #
    # rng_qe.random(n_hits_before) gives one random number in [0, 1)
    # for each hit.
    #
    # Hits with QE=1.0 are always kept because random number < 1.0.
    # Hits with QE=0.5 are kept about half the time.
    #
    keep = rng_qe.random(n_hits_before) < keep_probability

    n_removed = int(np.count_nonzero(~keep))
    n_removed_total += n_removed
    arr["ndigi_removed_by_QE"][i] = int(n_removed)

    # --------------------------------------------------------
    # Apply the same hit mask to all hit-level fields
    # --------------------------------------------------------
    for field in hit_level_fields:
        if field not in arr:
            continue

        arr_field = np.asarray(arr[field][i])

        # Only hit-level branches should have length n_hits_before.
        #
        # If a listed field does not match the hit multiplicity, stop,
        # because applying the mask would corrupt the event structure.
        if len(arr_field) != n_hits_before:
            raise ValueError(
                f"In event {i}, field '{field}' has length {len(arr_field)}, "
                f"but WCTE_pmt_id has length {n_hits_before}. "
                "This field is not consistent with the hit-level arrays."
            )

        arr[field][i] = arr_field[keep]

    # --------------------------------------------------------
    # Update event-level summary quantities
    # --------------------------------------------------------
    #
    # ndigi is the number of remaining digi-hits in this library event.
    # sumQ is recomputed after the hit removal.
    #
    arr["ndigi"][i] = int(np.count_nonzero(keep))
    arr["sumQ"][i] = float(np.sum(arr["q"][i])) if arr["ndigi"][i] > 0 else 0.0

    n_hits_after_total += int(arr["ndigi"][i])


print("QE hit removal at library-event level done.")
print(f"Total hits before QE:          {n_hits_before_total}")
print(f"Candidate hits on QE=0.5 PMTs: {n_qe05_hits_total}")
print(f"Total hits removed by QE:      {n_removed_total}")
print(f"Total hits after QE:           {n_hits_after_total}")

if n_qe05_hits_total > 0:
    print(
        "Removed fraction among QE=0.5 candidate hits:",
        n_removed_total / n_qe05_hits_total
    )


### Implementing SPE calibration

First, check the number of unique slot and pos pairs in the events and check if it matches with the calibrated slot-pos pairs

In [ ]:
# To check the number of unique slot-pos pairs in the library for debugging
# Flatten all event-level arrays
all_slots = np.concatenate([np.asarray(x) for x in arr["WCTE_slot"]])
all_pos   = np.concatenate([np.asarray(x) for x in arr["WCTE_pos"]])

# Build unique (slot, pos) pairs
pairs = np.column_stack([all_slots, all_pos])
unique_pairs = np.unique(pairs, axis=0)

print("Total hits:", len(pairs))
print("Unique (WCTE_slot, WCTE_pos) pairs:", len(unique_pairs))

In [ ]:
# ============================================================
# Load all SPE calibration chunks into one dictionary
# ============================================================

chunk_files = sorted(Path("/scratch/saborido/WCTE-AmBe-data/SPE_callibration/").glob("Final_run2306_770ps_chunk*.npz"))

if len(chunk_files) == 0:
    raise FileNotFoundError("No Final_run2306_770ps_chunk*.npz files found in the current directory.")

spe_rows = []

for f in chunk_files:
    with np.load(f, allow_pickle=True) as z:
        r = z["results"]
        spe_rows.append(r[["slot_id", "pos_id", "spe_mean", "gain"]])

spe_table = np.concatenate(spe_rows)

spe_dict = {
    "slot_id":  spe_table["slot_id"],
    "pos_id":   spe_table["pos_id"],
    "spe_mean": spe_table["spe_mean"],
    "gain":     spe_table["gain"],
}

# Fast lookup: (slot_id, pos_id) -> spe_mean / gain
spe_lookup = {
    (int(slot), int(pos)): {
        "spe_mean": float(spe),
        "gain": float(gain),
    }
    for slot, pos, spe, gain in zip(
        spe_dict["slot_id"],
        spe_dict["pos_id"],
        spe_dict["spe_mean"],
        spe_dict["gain"],
    )
}

spe_dict["lookup"] = spe_lookup

print(f"Loaded {len(chunk_files)} files")
print(f"Loaded {len(spe_table)} SPE calibration rows")
print(f"Unique (slot_id, pos_id) pairs: {len(spe_lookup)}")

In [ ]:

# ============================================================
# Add q_adc to arr using WCTE_slot/WCTE_pos matching
# ============================================================

q_adc = []
n_missing = 0

for q_evt, slot_evt, pos_evt in zip(arr["q"], arr["WCTE_slot"], arr["WCTE_pos"]):

    q_evt = np.asarray(q_evt, dtype=float)
    slot_evt = np.asarray(slot_evt)
    pos_evt = np.asarray(pos_evt)

    spe_evt = np.empty(len(q_evt), dtype=float)

    for i, slot, pos in zip(range(len(q_evt)), slot_evt, pos_evt):
        key = (int(slot), int(pos))

        if key in spe_lookup:
            spe_evt[i] = spe_lookup[key]["spe_mean"]
        else:
            spe_evt[i] = np.nan
            n_missing += 1

    q_adc.append(q_evt * spe_evt)

# Preserve jagged/event-by-event structure
arr["q_adc"] = np.asarray(q_adc, dtype=object)

# Event-level ADC charge sum
arr["sumQ_adc"] = np.asarray(
    [np.nansum(np.asarray(q_adc_evt, dtype=float)) for q_adc_evt in arr["q_adc"]],
    dtype=float,
)

print(f"Added arr['q_adc']")
print(f"Added arr['sumQ_adc']")
print(f"Missing SPE matches: {n_missing}")


In [ ]:
# ============================================================
# Remove hits with missing q_adc
# ============================================================
# Missing q_adc was encoded as np.nan in the previous cell.

hit_level_keys = [
    "tubeid",
    "mPMTid",
    "mPMT_pmtid",
    "q",
    "q_adc",
    "t",
    "x", "y", "z",
    "dx", "dy", "dz",
    "ntruth",
    "hit_from_prompt",
    "hit_from_capture",
    "hit_from_otherprocess",
    "WCTE_slot",
    "WCTE_pos",
    "WCTE_pmt_id",
]

# Event-by-event mask: True = keep hit, False = remove hit
hit_masks = [
    np.isfinite(np.asarray(q_adc_evt, dtype=float))
    for q_adc_evt in arr["q_adc"]
]

n_hits_before = sum(len(x) for x in arr["q_adc"])
n_hits_keep   = sum(np.count_nonzero(m) for m in hit_masks)
n_hits_remove = n_hits_before - n_hits_keep

# Apply the same mask to all hit-level variables
for key in hit_level_keys:
    arr[key] = np.asarray(
        [
            np.asarray(evt_values)[mask]
            for evt_values, mask in zip(arr[key], hit_masks)
        ],
        dtype=object,
    )

# Update ndigi because number of kept hits changed
if "ndigi" in arr:
    arr["ndigi"] = np.asarray([len(q_evt) for q_evt in arr["q"]], dtype=int)

# Optional: update sumQ after removing unmatched hits
if "sumQ" in arr:
    arr["sumQ"] = np.asarray([np.sum(q_evt) for q_evt in arr["q"]], dtype=float)

print("Removed hits with missing q_adc")
print(f"Hits before : {n_hits_before}")
print(f"Hits kept   : {n_hits_keep}")
print(f"Hits removed: {n_hits_remove}")

# Sanity check: q_adc should now have no NaN/inf
remaining_bad = sum(
    np.count_nonzero(~np.isfinite(np.asarray(q_adc_evt, dtype=float)))
    for q_adc_evt in arr["q_adc"]
)

print(f"Remaining non-finite q_adc values: {remaining_bad}")

## Build synthetic windows

The output is one pandas DataFrame row per window.

Each row stores variable-length NumPy arrays for the merged true-hit information in that window.

`make_window()` is a function that remembers things between calls:

- which window number it is currently generating
- which old events are still “alive” and may spill into the next window
- cached quantities computed once from the library

That remembered information is stored as attributes attached to the function itself:

- `make_window._state`

- `make_window._cache`

A helper function to clear the memory of the generator is needed: `def reset_make_window_stream()`

A helper function to inspect the state of the generator is defined: `get_make_window_status`

`hasattr(make_window, "_state")` checks whether the function already has a _state attribute.

`delattr(make_window, "_state")` deletes it if it exists.

The same is done for _cache.


This allows the generation of one window at a time while preserving a realistic source stream.

The shift operations are explicitly performed in float64. Otherwise, automatic casting to float32 can lead to a significant loss of precision when working with a very large number of windows, producing artifacts in the hit/capture times.

In [ ]:
def reset_make_window_stream():
    """Reset the internal state of the continuous window generator."""
    if hasattr(make_window, "_state"):
        delattr(make_window, "_state")
    if hasattr(make_window, "_cache"):
        delattr(make_window, "_cache")


def get_make_window_status():
    """
    Inspect the internal state of make_window without modifying it.
    """
    status = {
        "initialized": hasattr(make_window, "_state"),
        "has_cache": hasattr(make_window, "_cache"),
    }

    if hasattr(make_window, "_state"):
        state = make_window._state
        status["window_idx"] = state.get("window_idx", None)
        status["n_active"] = len(state.get("active", []))

        if len(state.get("active", [])) > 0:
            abs_ends = np.array([ev["abs_end"] for ev in state["active"]], dtype=float)
            shifts = np.array([ev["shift"] for ev in state["active"]], dtype=float)

            status["active_abs_end_min"] = float(abs_ends.min())
            status["active_abs_end_max"] = float(abs_ends.max())
            status["active_shift_min"] = float(shifts.min())
            status["active_shift_max"] = float(shifts.max())
        else:
            status["active_abs_end_min"] = None
            status["active_abs_end_max"] = None
            status["active_shift_min"] = None
            status["active_shift_max"] = None

    if hasattr(make_window, "_cache"):
        cache = make_window._cache
        status["cache_library_id"] = cache.get("library_id", None)

        rel_end_time = cache.get("rel_end_time", None)
        if rel_end_time is not None:
            status["rel_end_time_min"] = float(np.min(rel_end_time))
            status["rel_end_time_max"] = float(np.max(rel_end_time))
            status["rel_end_time_p99"] = float(np.quantile(rel_end_time, 0.99))

    return status


def make_window(library, activity_bq, deltat_ns, rng, start_idx, gap_ns=0.0):
    """
    Build one synthetic acquisition window from a continuous source stream.

    This function supports two cases:

    1) gap_ns = 0:
       Recorded windows are back-to-back.
       This reproduces the behavior of the current implementation.

    2) gap_ns > 0:
       There is a dead gap between recorded windows.
       The source still emits during the gap, but no data are recorded there.
       Decays that start in the gap can still produce delayed captures/hits
       inside the next recorded window.

    Main ideas
    ----------
    - New source decays are generated as a Poisson process.
    - Each library event is consumed without replacement only when it STARTS.
    - Once started, that event stays "active" until its latest relevant time
      (late hit or capture) has passed.
    - Only hits/captures falling inside the CURRENT RECORDED window are kept.

    Parameters
    ----------
    library : dict of awkward arrays / numpy arrays
        Event library loaded from wcte_ambe_hits.root
    activity_bq : float
        Source activity in Bq
    deltat_ns : float
        Live acquisition window length in ns
    rng : numpy.random.Generator
        Random number generator
    start_idx : int
        First unread library event to use for newly generated source decays
    gap_ns : float, optional
        Dead time between recorded windows, in ns.
        Default is 0.0, which gives the same behavior as the current code.

    Returns
    -------
    window_dict : dict
        Synthetic content of the current recorded window
    next_idx : int
        Next unread library index after consuming new decays introduced here
    """

    # ------------------------------------------------------------------
    # One-time cache:
    # For each library event, precompute
    #   1) the prompt relative time
    #   2) the latest relative time at which the event can still contribute
    #
    # This avoids recomputing event lifetimes on every window.
    # ------------------------------------------------------------------
    n_events_available = len(library["event"])

    cache_valid = (
        hasattr(make_window, "_cache")
        and make_window._cache.get("library_id", None) == id(library)
    )

    if not cache_valid:
        rel_end_time = np.zeros(n_events_available, dtype=float)
        prompt_rel_time = np.zeros(n_events_available, dtype=float)

        for i in range(n_events_available):
            # Largest relevant relative time inside event i.
            max_rel = 0.0

            if len(library["t"][i]) > 0:
                hit_times_i = np.asarray(library["t"][i], dtype=float)
                valid_hit_times_i = hit_times_i[hit_times_i < 1e9]  # this is introduced to don't carry forever the events with non-physical huge hit times

                if valid_hit_times_i.size > 0:
                    max_rel = max(max_rel, float(np.max(valid_hit_times_i)))

            if len(library["capture_t"][i]) > 0:
                cap_times_i = np.asarray(library["capture_t"][i], dtype=np.float64)
                max_rel = max(max_rel, float(np.max(cap_times_i)))  # for completness, but capture time is expected to be always smaller than the last hit time

            if int(library["has_prompt"][i]) == 1:
                prompt_rel_time[i] = float(library["prompt_track_time"][i])
                max_rel = max(max_rel, prompt_rel_time[i])  # for completness, but prompt_rel_time is often 0

            rel_end_time[i] = max_rel

        make_window._cache = {
            "library_id": id(library),
            "rel_end_time": rel_end_time,
            "prompt_rel_time": prompt_rel_time,
        }

    rel_end_time = make_window._cache["rel_end_time"]
    prompt_rel_time = make_window._cache["prompt_rel_time"]

    # ------------------------------------------------------------------
    # Internal state of the continuous stream
    #
    # window_idx counts how many RECORDED windows have already been produced.
    # active stores events that started earlier and may still contribute now.
    # ------------------------------------------------------------------
    if not hasattr(make_window, "_state"):
        make_window._state = {
            "window_idx": 0,
            "active": [],   # each element: {"lib_idx", "shift", "abs_end"}
        }

    state = make_window._state
    window_idx = state["window_idx"]

    # ------------------------------------------------------------------
    # Absolute timing of the current RECORDED window
    #
    # For gap_ns = 0:
    #   t_start = window_idx * deltat_ns
    #   t_end   = t_start + deltat_ns
    #
    # For gap_ns > 0:
    #   recorded windows are separated by a dead gap
    #   stride = live window + dead gap
    # ------------------------------------------------------------------
    stride_ns = deltat_ns + gap_ns
    t_start = window_idx * stride_ns
    t_end = t_start + deltat_ns

    # ------------------------------------------------------------------
    # Time interval over which NEW source decays must be generated on this call
    #
    # If gap_ns = 0:
    #   gen interval = [t_start, t_end]
    #   which is exactly the same as the current implementation
    #
    # If gap_ns > 0:
    #   gen interval = [t_start - gap_ns, t_end]
    #   meaning:
    #     - decays during the dead gap before this window are generated now
    #     - decays during the current live window are also generated now
    #
    # This lets prompt-less delayed captures from the gap appear naturally
    # at the beginning of the recorded window.
    # ------------------------------------------------------------------
    gen_start = t_start - gap_ns
    gen_end = t_end
    gen_dt_ns = gen_end - gen_start
    gen_dt_s = gen_dt_ns * 1e-9

    n_overlay = rng.poisson(activity_bq * gen_dt_s)

    end_idx = start_idx + n_overlay
    if end_idx > n_events_available:
        raise ValueError(
            f"Not enough events left in library: requested {n_overlay} starting at {start_idx}, "
            f"but library size is {n_events_available}."
        )

    chosen = np.arange(start_idx, end_idx, dtype=np.int32)

    # ------------------------------------------------------------------
    # Add newly started events to the active list
    #
    # For each newly generated source decay:
    #   - choose its absolute prompt time uniformly in the generation interval
    #   - compute the shift between library-relative time and absolute time
    #   - compute the absolute end time of the event
    #
    # Later, only the pieces falling inside [t_start, t_end] are kept.
    # ------------------------------------------------------------------
    for lib_idx in chosen:
        prompt_abs_t = np.float64(rng.uniform(gen_start, gen_end))

        # General mapping:
        #   absolute_time = relative_time + shift
        shift = np.float64(prompt_abs_t - prompt_rel_time[lib_idx])

        # Latest absolute time at which this event can still contribute
        abs_end = np.float64(shift + rel_end_time[lib_idx])

        state["active"].append(
            {
                "lib_idx": int(lib_idx),
                "shift": float(shift),
                "abs_end": float(abs_end),
            }
        )

    # ------------------------------------------------------------------
    # Collect everything from active events that falls in the CURRENT
    # RECORDED window [t_start, t_end].
    # ------------------------------------------------------------------
    all_hit_t = []
    all_hit_q = []
    all_hit_q_adc = []
    all_tubeid = []
    all_mPMTid = []
    all_mPMT_pmtid = []

    # WCTE hit-level IDs, already added to arr before window generation
    all_WCTE_slot = []
    all_WCTE_pos = []
    all_WCTE_pmt_id = []

    all_x = []
    all_y = []
    all_z = []
    all_dx = []
    all_dy = []
    all_dz = []
    all_hit_from_prompt = []
    all_hit_from_capture = []
    all_source_event_idx = []
    all_source_event = []
    all_source_subevent = []

    all_capture_t = []
    all_relative_capture_t = []
    all_prompt_time_withcapture = []
    all_prompt_time = []
    all_capture_x = []
    all_capture_y = []
    all_capture_z = []
    all_capture_nucleus = []
    all_capture_ngamma = []
    all_capture_total_gammaE = []

    next_active = []

    for ev in state["active"]:
        lib_idx = ev["lib_idx"]
        shift = ev["shift"]
        abs_end = ev["abs_end"]

        # Keep event active for the NEXT call only if it can still contribute
        # after the end of the current recorded window.
        if abs_end > t_end:
            next_active.append(ev)

        # --------------------------------------------------------------
        # Hits in the current recorded window
        # --------------------------------------------------------------
        hit_rel_t = np.asarray(library["t"][lib_idx], dtype=np.float64)
        hit_abs_t = hit_rel_t + np.float64(shift)
        keep = (hit_abs_t >= t_start) & (hit_abs_t <= t_end)

        if np.any(keep):
            # Store hit times relative to the START of the recorded window
            all_hit_t.append(hit_abs_t[keep] - t_start)

            all_hit_q.append(library["q"][lib_idx][keep])
            all_hit_q_adc.append(library["q_adc"][lib_idx][keep])
            all_tubeid.append(library["tubeid"][lib_idx][keep])
            all_mPMTid.append(library["mPMTid"][lib_idx][keep])
            all_mPMT_pmtid.append(library["mPMT_pmtid"][lib_idx][keep])
            all_WCTE_slot.append(library["WCTE_slot"][lib_idx][keep])
            all_WCTE_pos.append(library["WCTE_pos"][lib_idx][keep])
            all_WCTE_pmt_id.append(library["WCTE_pmt_id"][lib_idx][keep])
            all_x.append(library["x"][lib_idx][keep])
            all_y.append(library["y"][lib_idx][keep])
            all_z.append(library["z"][lib_idx][keep])
            all_dx.append(library["dx"][lib_idx][keep])
            all_dy.append(library["dy"][lib_idx][keep])
            all_dz.append(library["dz"][lib_idx][keep])
            all_hit_from_prompt.append(library["hit_from_prompt"][lib_idx][keep])
            all_hit_from_capture.append(library["hit_from_capture"][lib_idx][keep])

            n_kept = int(np.count_nonzero(keep))
            all_source_event_idx.append(np.full(n_kept, lib_idx, dtype=np.int32))
            all_source_event.append(np.full(n_kept, library["event"][lib_idx], dtype=np.int32))
            all_source_subevent.append(np.full(n_kept, library["subevent"][lib_idx], dtype=np.int32))

        # --------------------------------------------------------------
        # Prompt truth in the current recorded window
        #
        # If the prompt happened in the gap or an earlier window, it is not
        # stored here, even if delayed capture products from the same event
        # are recorded now.
        # --------------------------------------------------------------
        if int(library["has_prompt"][lib_idx]) == 1:
            prompt_abs_t = np.float64(prompt_rel_time[lib_idx]) + np.float64(shift)
            prompt_keep_all = (prompt_abs_t >= t_start) and (prompt_abs_t <= t_end)

            if prompt_keep_all:
                all_prompt_time.append(np.array([prompt_abs_t - t_start], dtype=float))

        # --------------------------------------------------------------
        # Capture truth in the current recorded window
        # --------------------------------------------------------------
        if library["ncaptures"][lib_idx] > 0:
            cap_rel_t = np.asarray(library["capture_t"][lib_idx], dtype=np.float64)
            cap_abs_t = cap_rel_t + np.float64(shift)
            cap_keep = (cap_abs_t >= t_start) & (cap_abs_t <= t_end)

            if np.any(cap_keep):
                all_capture_t.append(cap_abs_t[cap_keep] - t_start)
                all_relative_capture_t.append(library["capture_t"][lib_idx][cap_keep])

                # For prompt_time_withcapture, only store the prompt if that
                # prompt itself is also inside the recorded window.
                if int(library["has_prompt"][lib_idx]) == 1:
                    prompt_abs_t = np.float64(prompt_rel_time[lib_idx]) + np.float64(shift)
                    prompt_keep = (
                        (prompt_abs_t >= t_start)
                        and (prompt_abs_t <= t_end)
                        and np.all(prompt_rel_time[lib_idx] < library["capture_t"][lib_idx][cap_keep])
                    )

                    if prompt_keep:
                        all_prompt_time_withcapture.append(
                            np.full(np.count_nonzero(cap_keep), prompt_abs_t - t_start, dtype=float)
                        )

                all_capture_x.append(library["capture_x"][lib_idx][cap_keep])
                all_capture_y.append(library["capture_y"][lib_idx][cap_keep])
                all_capture_z.append(library["capture_z"][lib_idx][cap_keep])
                all_capture_nucleus.append(library["capture_nucleus"][lib_idx][cap_keep])
                all_capture_ngamma.append(library["capture_ngamma"][lib_idx][cap_keep])
                all_capture_total_gammaE.append(library["capture_total_gammaE"][lib_idx][cap_keep])

    # ------------------------------------------------------------------
    # Update the stream state for the next call
    # ------------------------------------------------------------------
    state["active"] = next_active
    state["window_idx"] += 1

    # ------------------------------------------------------------------
    # Helper to concatenate per-event contributions into one window array
    # ------------------------------------------------------------------
    def _cat(lst, dtype):
        if not lst:
            return np.array([], dtype=dtype)
        return np.concatenate(lst).astype(dtype, copy=False)

    hit_t = _cat(all_hit_t, float)
    hit_q = _cat(all_hit_q, np.float32)
    hit_q_adc = _cat(all_hit_q_adc, np.float32)
    tubeid = _cat(all_tubeid, np.int32)
    mPMTid = _cat(all_mPMTid, np.int32)
    mPMT_pmtid = _cat(all_mPMT_pmtid, np.int32)
    WCTE_slot = _cat(all_WCTE_slot, np.int32)
    WCTE_pos = _cat(all_WCTE_pos, np.int32)
    WCTE_pmt_id = _cat(all_WCTE_pmt_id, np.int32)
    x = _cat(all_x, np.float32)
    y = _cat(all_y, np.float32)
    z = _cat(all_z, np.float32)
    dx = _cat(all_dx, np.float32)
    dy = _cat(all_dy, np.float32)
    dz = _cat(all_dz, np.float32)
    hit_from_prompt = _cat(all_hit_from_prompt, np.int32)
    hit_from_capture = _cat(all_hit_from_capture, np.int32)
    source_event_idx = _cat(all_source_event_idx, np.int32)
    source_event = _cat(all_source_event, np.int32)
    source_subevent = _cat(all_source_subevent, np.int32)

    capture_t = _cat(all_capture_t, float)
    relative_capture_t = _cat(all_relative_capture_t, float)
    prompt_time_withcapture = _cat(all_prompt_time_withcapture, float)
    prompt_time = _cat(all_prompt_time, float)
    capture_x = _cat(all_capture_x, np.float32)
    capture_y = _cat(all_capture_y, np.float32)
    capture_z = _cat(all_capture_z, np.float32)
    capture_nucleus = _cat(all_capture_nucleus, np.int32)
    capture_ngamma = _cat(all_capture_ngamma, np.int32)
    capture_total_gammaE = _cat(all_capture_total_gammaE, np.float32)

    # ------------------------------------------------------------------
    # Sort hit-level arrays by hit time inside the recorded window
    # ------------------------------------------------------------------
    if hit_t.size > 0:
        order = np.argsort(hit_t, kind="stable")
        hit_t = hit_t[order]
        hit_q = hit_q[order]
        hit_q_adc = hit_q_adc[order]
        tubeid = tubeid[order]
        mPMTid = mPMTid[order]
        mPMT_pmtid = mPMT_pmtid[order]
        WCTE_slot = WCTE_slot[order]
        WCTE_pos = WCTE_pos[order]
        WCTE_pmt_id = WCTE_pmt_id[order]
        x = x[order]
        y = y[order]
        z = z[order]
        dx = dx[order]
        dy = dy[order]
        dz = dz[order]
        hit_from_prompt = hit_from_prompt[order]
        hit_from_capture = hit_from_capture[order]
        source_event_idx = source_event_idx[order]
        source_event = source_event[order]
        source_subevent = source_subevent[order]

    return {
        # Number of NEW library events introduced on this call.
        # For gap_ns = 0, this means new decays started in the current window.
        # For gap_ns > 0, this means new decays started in the interval
        # [t_start - gap_ns, t_end].
        "n_overlay": int(n_overlay),

        "window_sumQ": float(hit_q.sum()) if hit_q.size else 0.0,
        "window_sumQ_adc": float(hit_q_adc.sum()) if hit_q_adc.size else 0.0,
        "n_hits": int(hit_t.size),
        "hit_t": hit_t,
        "hit_q": hit_q,
        "hit_q_adc": hit_q_adc,
        "tubeid": tubeid,
        "mPMTid": mPMTid,
        "mPMT_pmtid": mPMT_pmtid,
        "WCTE_slot": WCTE_slot,
        "WCTE_pos": WCTE_pos,
        "WCTE_pmt_id": WCTE_pmt_id,
        "x": x,
        "y": y,
        "z": z,
        "dx": dx,
        "dy": dy,
        "dz": dz,
        "hit_from_prompt": hit_from_prompt,
        "hit_from_capture": hit_from_capture,
        "source_event_idx": source_event_idx,
        "source_event": source_event,
        "source_subevent": source_subevent,
        "capture_t": capture_t,
        "relative_capture_t": relative_capture_t,
        "prompt_time_withcapture": prompt_time_withcapture,
        "prompt_time": prompt_time,
        "capture_x": capture_x,
        "capture_y": capture_y,
        "capture_z": capture_z,
        "capture_nucleus": capture_nucleus,
        "capture_ngamma": capture_ngamma,
        "capture_total_gammaE": capture_total_gammaE,
    }, end_idx

In [ ]:
# Generate many synthetic windows from one continuous run
# and throw away the first 5 warmup windows

reset_make_window_stream()

warmup_windows = 5
windows = []
next_idx = 0

for i in range(N_windows + warmup_windows):
    w, next_idx = make_window(arr, A, deltat_ns, rng, start_idx=next_idx, gap_ns=dead_time_ns)

    # Discard the first warmup_windows windows to warm up the left edge
    if i < warmup_windows:
        continue

    kept_i = i - warmup_windows
    w["window_id"] = kept_i
    windows.append(w)

    if (kept_i + 1) % 1000 == 0:
        print(f"Generated {kept_i + 1} / {N_windows} windows...")
        
        s = get_make_window_status()
        print(
            f"{i+1} windows | "
            f"window_idx={s.get('window_idx')} | "
            f"n_active={s.get('n_active')} | "
            #f"rel_end_time_max={s.get('rel_end_time_max')}"
        )
        print(s)

In [ ]:
windows_df = pd.DataFrame(windows)
pd.set_option('display.max_columns', None)
windows_df.head()

In [ ]:
# Summary
print(f"Total number of events used: {windows_df['n_overlay'].sum()} / {n_library_events}")

summary = windows_df[["n_overlay", "n_hits", "window_sumQ"]].copy()
summary.describe()

In [ ]:
# Inspect one synthetic window
k = 5
row = windows_df.iloc[k]

print(f"window_id = {row['window_id']}")
print(f"n_overlay = {row['n_overlay']}")
print(f"n_hits = {row['n_hits']}")
print(f"window_sumQ = {row['window_sumQ']:.3f}")
print(f"prompt times [ns] = {row['prompt_time']}")
print("absolute capture_t [ns] =", row["capture_t"])
print("capture time from prompt (relative) [ns] =", row["relative_capture_t"])

print("True-hit times [ns] =", *row["hit_t"])
print("Hit charges =", *row["hit_q"])
print("Is from prompt =", *row["hit_from_prompt"])
print(f"Len of capture_t = {len(row['capture_t'])}")

# mask = (row["hit_t"] > 250_000) & (row["hit_t"] < 255_000) # for debugging
#print("True-hit times [ns] =", *row["hit_t"][mask])
#print("Hit charges =", *row["hit_q"][mask])
#print("Is from prompt =", *row["hit_from_prompt"][mask])
#print(f"Len of capture_t = {len(row['capture_t'])}")

In [ ]:
# Flatten one synthetic window into a per-hit table
k = 0
row = windows_df.iloc[k]

hits_df = pd.DataFrame({
    "tubeid": row["tubeid"],
    "mPMTid": row["mPMTid"],
    "mPMT_pmtid": row["mPMT_pmtid"],
    "x": row["x"],
    "y": row["y"],
    "z": row["z"],
    "dx": row["dx"],
    "dy": row["dy"],
    "dz": row["dz"],

    "hit_t": row["hit_t"],
    "hit_q": row["hit_q"],
    "hit_from_prompt": row["hit_from_prompt"],
    "hit_from_capture": row["hit_from_capture"],
    "source_event_idx": row["source_event_idx"],
})

hits_df.head(10)



## Note: Resolution effect!

- This dataframe does **not** merge multiple hits on the same PMT after overlay. It overlays and sorts them. The resolution effect is applied after signal and background merge in `merge_SB.py`


## Descriptive plots

In [ ]:
window_idx = 76
w = windows[window_idx]

mask_prompts = w["hit_from_prompt"] == 1
mask_captures = w["hit_from_capture"] == 1
prompt_times = w["hit_t"][mask_prompts]
capture_times = w["hit_t"][mask_captures]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(prompt_times, bins=np.linspace(0,270_000,1001), alpha=0.5, color="blue", label="Prompt hits")
ax.hist(capture_times, bins=np.linspace(0,270_000,1001), alpha=0.5, color="red", label="Capture hits")
ax.set_xlabel("Hit time [ns]", fontsize=14)
ax.set_ylabel("Counts", fontsize=14)
ax.set_yscale("log")
ax.set_title(f"Hit times distribution (Window {window_idx})\n N hits: {len(w['hit_t'])}, N captures: {len(w['capture_t'])}", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Build flattened arrays
all_hit_t = np.concatenate([w for w in windows_df["hit_t"] if len(w) > 0]) if len(windows_df) else np.array([])
all_hit_q = np.concatenate([w for w in windows_df["hit_q"] if len(w) > 0]) if len(windows_df) else np.array([])
all_hit_q_adc = np.concatenate([w for w in windows_df["hit_q_adc"] if len(w) > 0]) if len(windows_df) else np.array([])
all_capture_t = np.concatenate([w for w in windows_df["capture_t"] if len(w) > 0]) if len(windows_df) else np.array([])
all_relative_capture_t = np.concatenate([w for w in windows_df["relative_capture_t"] if len(w) > 0]) if len(windows_df) else np.array([])
all_prompt_time = np.concatenate([w for w in windows_df["prompt_time"] if len(w) > 0]) if len(windows_df) else np.array([])
all_capture_total_gammaE = np.concatenate([w for w in windows_df["capture_total_gammaE"] if len(w) > 0]) if len(windows_df) else np.array([])

# Number of captures kept per window
n_captures_kept = windows_df["capture_t"].apply(len).to_numpy()

# Choose one example window to visualize in time
window_idx = 76
w = windows[window_idx]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# 1) Overlay multiplicity
axes[0, 0].hist(
    windows_df["n_overlay"],
    bins=np.arange(-0.5, np.max(windows_df["n_overlay"]) + 1.5, 1),
    color="teal")
axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel("Number of source decays per window", fontsize=14)
axes[0, 0].set_ylabel("Windows", fontsize=14)
axes[0, 0].set_title("Overlay multiplicity", fontsize=16)
axes[0, 0].tick_params(axis="both", labelsize=12)

# 2) Hits per window
axes[0, 1].hist(windows_df["n_hits"], bins=np.linspace(1, 500, 501), color="teal")
axes[0, 1].set_xlabel("Number of digi hits per window", fontsize=14)
axes[0, 1].set_ylabel("Windows", fontsize=14)
axes[0, 1].set_title("Hits per window", fontsize=16)
axes[0, 1].tick_params(axis="both", labelsize=12)

# 3) Total charge per window
axes[0, 2].hist(windows_df["window_sumQ"], bins=np.linspace(1,1000,1001), color="teal")
axes[0, 2].set_xlabel(r"Total integrated charge per window [p.e.]", fontsize=14)
axes[0, 2].set_ylabel("Windows", fontsize=14)
axes[0, 2].set_title("Total charge per window", fontsize=16)
axes[0, 2].tick_params(axis="both", labelsize=12)

# 4) All hit times
axes[1, 0].hist(all_hit_t, bins=100, range=(0, deltat_ns), color="teal")
axes[1, 0].set_xlabel("Hit time within window [ns]", fontsize=14)
axes[1, 0].set_ylabel("Counts", fontsize=14)
axes[1, 0].set_title("All hit times", fontsize=16)
axes[1, 0].tick_params(axis="both", labelsize=12)

# 5) All hit charges
axes[1, 1].hist(all_hit_q, bins=2000, range=(0, np.max(all_hit_q)), color="teal")
axes[1, 1].set_xlim(0, 6)
axes[1, 1].set_xlabel("Hit charge [p.e.]", fontsize=14)
axes[1, 1].set_ylabel("Counts", fontsize=14)
axes[1, 1].set_title("All hit charges", fontsize=16)
axes[1, 1].tick_params(axis="both", labelsize=12)

'''
# 5) Hits vs overlays
axes[1, 1].scatter(
    windows_df["n_overlay"],
    windows_df["n_hits"],
    color="teal",
    s=10,
    alpha=0.5
)
axes[1, 1].set_xlabel("Number of source decays per window", fontsize=14)
axes[1, 1].set_ylabel("Number of true hits per window", fontsize=14)
axes[1, 1].set_title("Hits vs overlays", fontsize=16)
axes[1, 1].tick_params(axis="both", labelsize=12)
'''

# 6) Example window: hit charge vs time with capture markers
prompt_times = w["prompt_time"]

decay_ids = np.asarray(w["source_event_idx"])
unique_decay_ids = np.unique(decay_ids)

cmap = plt.get_cmap("tab20", max(len(unique_decay_ids), 1))

for i, decay_id in enumerate(unique_decay_ids):
    mask = decay_ids == decay_id

    axes[1, 2].scatter(
        w["hit_t"][mask],
        w["hit_q_adc"][mask],
        s=6,
        alpha=0.7,
        label="Hits" if i == 0 else None,
        color=cmap(i)
    )

for tc in prompt_times:
    axes[1, 2].axvline(
        tc,
        linestyle="--",
        linewidth=1,
        label="Prompt" if tc == prompt_times[0] else None,
        color="blue",
        alpha=0.7
    )
for tc in w["capture_t"]:
    axes[1, 2].axvline(
        tc,
        linestyle="--",
        linewidth=1,
        label="Capture" if tc == w["capture_t"][0] else None,
        color="red",
        alpha=0.7
    )
axes[1, 2].set_xlabel("Time within window [ns]", fontsize=14)
axes[1, 2].set_xlim(0, deltat_ns)
axes[1, 2].set_ylabel("Hit charge [ADC]", fontsize=14)
axes[1, 2].set_title(
    f"Example window {window_idx}\nNumber of captures: {len(w['capture_t'])}",
    fontsize=16
)
axes[1, 2].tick_params(axis="both", labelsize=12)
axes[1, 2].legend(fontsize=12)

plt.tight_layout()
plt.show()

# Summary focused on captures
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(n_captures_kept, bins=np.arange(-0.5, np.max(n_captures_kept) + 1.5, 1), color="teal")
axes[0].set_yscale("log")
axes[0].set_xlabel("Captures kept per window", fontsize=14)
axes[0].set_ylabel("Windows", fontsize=14)
axes[0].set_title("Capture multiplicity", fontsize=16)
axes[0].tick_params(axis="both", labelsize=12)

axes[1].hist(all_capture_total_gammaE, bins=100, color="teal")
axes[1].set_xlabel("Total capture gamma energy [MeV]", fontsize=14)
axes[1].set_ylabel("Counts", fontsize=14)
axes[1].set_yscale("log")
axes[1].set_title("Capture gamma energy", fontsize=16)
axes[1].tick_params(axis="both", labelsize=12)


axes[2].hist(all_relative_capture_t, bins=100, range=(0, deltat_ns), label = "Capture time - prompt time", alpha=0.7, color="teal")
axes[2].hist(all_capture_t, bins=100, range=(0, deltat_ns), label = "Absolute capture time", alpha=0.7, color="darkslategray")
axes[2].set_xlabel("Capture time [ns]", fontsize=14)
axes[2].set_ylabel("Counts", fontsize=14)
axes[2].set_title("Capture time", fontsize=16)
axes[2].tick_params(axis="both", labelsize=12)
axes[2].legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram of the prompt times

prompt_times = all_prompt_time
prompt_times_captures = np.concatenate([w for w in windows_df["prompt_time_withcapture"] if len(w) > 0]) if len(windows_df) else np.array([])

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(
    prompt_times_captures,
    bins=100,
    range=(0, deltat_ns),
    label="Prompts with capture in the same window",
    color="teal",
    histtype="step",
    linewidth=2,
    linestyle="-",
)
ax.hist(
    prompt_times,
    bins=100,
    range=(0, deltat_ns),
    label="All prompts",
    color="darkslategray",
    histtype="step",
    linewidth=2,
    linestyle="--",
)
ax.set_xlabel("Prompt time [ns]", fontsize=14)
ax.set_ylabel("Counts", fontsize=14)
ax.set_title("Prompt times", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------
# Flatten capture info
# ---------------------------------
all_capture_nucleus = (
    np.concatenate([w for w in windows_df["capture_nucleus"] if len(w) > 0])
    if len(windows_df) else np.array([])
)

all_capture_x = (
    np.concatenate([w for w in windows_df["capture_x"] if len(w) > 0])
    if len(windows_df) else np.array([])
)
all_capture_y = (
    np.concatenate([w for w in windows_df["capture_y"] if len(w) > 0])
    if len(windows_df) else np.array([])
)
all_capture_z = (
    np.concatenate([w for w in windows_df["capture_z"] if len(w) > 0])
    if len(windows_df) else np.array([])
)

# ---------------------------------
# Nucleus ID -> label map
# ---------------------------------
nucleus_labels = {
    1000010020: "H-2",
    1000080170: "O-17",
    1000080180: "O-18",
    1000080190: "O-19",
    1000280590: "Ni-59",
    1000320710: "Ge-71",
    1000320730: "Ge-73",
    1000320740: "Ge-74",
    1000320750: "Ge-75",
    1000320770: "Ge-77",
    1000641530: "Gd-153",
    1000641550: "Gd-155",
    1000641560: "Gd-156",
    1000641570: "Gd-157",
    1000641580: "Gd-158",
    1000641590: "Gd-159",
    1000641610: "Gd-161",
    1000832100: "Bi-210",
}

# Count captures per nucleus
unique_nuclei, counts_nuclei = np.unique(all_capture_nucleus, return_counts=True)

# ---------------------------------
# Choose nuclei to plot
# ---------------------------------
# Examples:
# selected_nuclei = None                     # all nuclei
# selected_nuclei = [1000010020]             # only H capture
# selected_nuclei = [1000641550, 1000641570] # some Gd isotopes

selected_nuclei = None

if selected_nuclei is None:
    position_mask = np.ones_like(all_capture_nucleus, dtype=bool)
    selection_label = "All captures"
else:
    selected_nuclei = np.array(selected_nuclei)
    position_mask = np.isin(all_capture_nucleus, selected_nuclei)
    selection_label = ", ".join(
        nucleus_labels.get(int(n), str(int(n))) for n in selected_nuclei
    )

# Filtered arrays
capture_x_sel = all_capture_x[position_mask]
capture_y_sel = all_capture_y[position_mask]
capture_z_sel = all_capture_z[position_mask]

# ---------------------------------
# 1) Histogram / bar plot of capture nucleus
# ---------------------------------
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(np.arange(len(unique_nuclei)), counts_nuclei, color="teal")
ax.set_xlabel("Capture nucleus", fontsize=14)
ax.set_ylabel("Counts", fontsize=14)
ax.set_yscale("log")
ax.set_title("Number of captures by nucleus", fontsize=16)
ax.set_xticks(np.arange(len(unique_nuclei)))
ax.set_xticklabels(
    [nucleus_labels.get(int(n), str(int(n))) for n in unique_nuclei],
    rotation=45,
    ha="right",
    fontsize=11,
)
ax.tick_params(axis="y", labelsize=12)
plt.tight_layout()
plt.show()

# ---------------------------------
# 2) Total capture gamma energy split by nucleus
# ---------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

for nucleus in unique_nuclei:
    mask = all_capture_nucleus == nucleus
    ax.hist(
        all_capture_total_gammaE[mask],
        bins=100,
        histtype="step",
        linewidth=1.8,
        label=nucleus_labels.get(int(nucleus), str(int(nucleus))),
    )

ax.set_xlabel("Total capture gamma energy [MeV]", fontsize=14)
ax.set_ylabel("Counts", fontsize=14)
ax.set_yscale("log")
ax.set_title("Capture gamma energy by nucleus", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(title="Nucleus", fontsize=10)
plt.tight_layout()
plt.show()

# ---------------------------------
# 3) Capture position, filtered by nucleus
# ---------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(capture_x_sel, bins=100, alpha=0.7, label="x", color="teal")
ax.hist(capture_y_sel, bins=100, alpha=0.7, label="y", color="darkslategray")
ax.hist(capture_z_sel, bins=100, alpha=0.7, label="z", color="lightcoral")

ax.set_xlabel("Capture position [cm]", fontsize=14)
ax.set_ylabel("Counts", fontsize=14)
ax.set_title(f"Capture positions ({selection_label})", fontsize=16)
ax.tick_params(axis="both", labelsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Unique capture nuclei: {unique_nuclei}, counts: {counts_nuclei}")
print(f"Percentage of captures per nucleus: {counts_nuclei / counts_nuclei.sum() * 100}")

if selected_nuclei is not None:
    print(f"Selected nuclei: {selected_nuclei}")
    print(
        "Selected nuclei labels:",
        [nucleus_labels.get(int(n), str(int(n))) for n in selected_nuclei],
    )
    print(f"Number of selected captures: {position_mask.sum()}")

### ID List

- **1000010020** — **H-2 / deuterium**
- **1000010030** — **H-3 / tritium**
- **1000080170** — **O-17**
- **1000320710** — **Ge-71 (germanium-71)**
- **1000641530** — **Gd-153**
- **1000641550** — **Gd-155**
- **1000641560** — **Gd-156**
- **1000641570** — **Gd-157**
- **1000641580** — **Gd-158**
- **1000641590** — **Gd-159**
- **1000641610** — **Gd-161**
- **1000832100** — **Bi-210 (bismuth-210)**

---
---
---

Save the data back to a .root replicating real data format.

## Tree: `WCTEReadoutWindows`

### Empty branches

* `window_time`
* `start_counter`
* `run_id`
* `sub_run_id`
* `spill_counter`
* `readout_number`
* `trigger_types`
* `trigger_times`
* `led_gains`
* `led_dacsettings`
* `led_ids`
* `led_card_ids`
* `led_slot_numbers`
* `led_event_types`
* `led_types`
* `led_sequence_numbers`
* `led_counters`
* `pmt_waveform_mpmt_card_ids`
* `pmt_waveform_pmt_channel_ids`
* `pmt_waveform_mpmt_slot_ids`
* `pmt_waveform_pmt_position_ids`
* `pmt_waveform_times`
* `pmt_waveforms`
* `beamline_pmt_qdc_charges`
* `beamline_pmt_tdc_times`
* `beamline_pmt_qdc_ids`
* `beamline_pmt_tdc_ids`

### Filled branches

* `event_number` ← `window_id`
* `hit_pmt_times` ← `hit_t`
* `hit_pmt_calibrated_times` ← `hit_t`
* `hit_pmt_charges` ← `hit_q`
* `hit_mpmt_slot_ids` ← `WCTE_slot`
* `hit_pmt_position_ids` ← `WCTE_pos`
* `hit_mpmt_card_ids` ← same-length list as `hit_mpmt_slot_ids`, filled with `0`
* `hit_pmt_channel_ids`← same-length list as `hit_mpmt_position_ids`, filled with `0`
* `hit_pmt_has_time_constant` ← same-length list as hits, filled with `1`


## Tree: `TTrueInfo`

* `event_number` ← `window_id`
* `n_overlay`
* `x`
* `y`
* `z`
* `dx`
* `dy`
* `dz`
* `hit_from_prompt`
* `hit_from_capture`
* `source_event_idx`
* `capture_t`
* `relative_capture_t`
* `prompt_time_withcapture`
* `prompt_time`
* `capture_x`
* `capture_y`
* `capture_z`
* `capture_nucleus`
* `capture_ngamma`
* `capture_total_gammaE`


In [ ]:
import awkward as ak
import gc
from tqdm.auto import tqdm

# ------------------------------------------------------------
# Output path
# ------------------------------------------------------------
output_root = Path("output/wcte_ambe_mc_digidata_0.root")
output_root.parent.mkdir(parents=True, exist_ok=True)

n_windows = len(windows_df)

# Smaller = smoother progress + lower RAM, but slower.
# Larger = faster, but more RAM.
chunk_size = 2000

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def series_to_ak(series, dtype=None):
    out = []
    for x in series:
        arr = np.asarray(x)
        if dtype is not None:
            arr = arr.astype(dtype, copy=False)
        out.append(arr)
    return ak.Array(out)


def empty_jagged(n, dtype):
    """
    Fast empty jagged array:
    creates n empty lists without making n separate numpy arrays.
    """
    offsets = ak.index.Index64(np.zeros(n + 1, dtype=np.int64))
    content = ak.contents.NumpyArray(np.empty(0, dtype=dtype))
    return ak.Array(ak.contents.ListOffsetArray(offsets, content))


def constant_like(series, value, dtype):
    out = []
    for x in series:
        arr = np.asarray(x)
        out.append(np.full(len(arr), value, dtype=dtype))
    return ak.Array(out)


# ------------------------------------------------------------
# Build one WCTEReadoutWindows chunk
# ------------------------------------------------------------
def build_wcte_tree(df):
    n = len(df)

    zeros_i32 = np.zeros(n, dtype=np.int32)
    zeros_i64 = np.zeros(n, dtype=np.int64)

    event_number = df["window_id"].to_numpy(dtype=np.int32)

    return {
        "window_time": zeros_i64,
        "start_counter": zeros_i64,
        "run_id": zeros_i32,
        "sub_run_id": zeros_i32,
        "spill_counter": zeros_i32,
        "event_number": event_number,
        "readout_number": zeros_i32,

        "trigger_types": empty_jagged(n, np.int32),
        "trigger_times": empty_jagged(n, np.float64),
        "led_gains": empty_jagged(n, np.float32),
        "led_dacsettings": empty_jagged(n, np.int32),
        "led_ids": empty_jagged(n, np.int32),
        "led_card_ids": empty_jagged(n, np.int32),
        "led_slot_numbers": empty_jagged(n, np.int32),
        "led_event_types": empty_jagged(n, np.int32),
        "led_types": empty_jagged(n, np.int32),
        "led_sequence_numbers": empty_jagged(n, np.int32),
        "led_counters": empty_jagged(n, np.int32),

        "hit_mpmt_card_ids": constant_like(df["WCTE_slot"], 0, np.int32),
        "hit_pmt_channel_ids": constant_like(df["WCTE_pos"], 0, np.int32),
        "hit_mpmt_slot_ids": series_to_ak(df["WCTE_slot"], np.int32),
        "hit_pmt_position_ids": series_to_ak(df["WCTE_pos"], np.int32),
        "hit_pmt_charges": series_to_ak(df["hit_q"], np.float32),
        "hit_pmt_times": series_to_ak(df["hit_t"], np.float64),

        "pmt_waveform_mpmt_card_ids": empty_jagged(n, np.int32),
        "pmt_waveform_pmt_channel_ids": empty_jagged(n, np.int32),
        "pmt_waveform_mpmt_slot_ids": empty_jagged(n, np.int32),
        "pmt_waveform_pmt_position_ids": empty_jagged(n, np.int32),
        "pmt_waveform_times": empty_jagged(n, np.float64),
        "pmt_waveforms": empty_jagged(n, np.float32),

        "beamline_pmt_qdc_charges": empty_jagged(n, np.float32),
        "beamline_pmt_tdc_times": empty_jagged(n, np.float64),
        "beamline_pmt_qdc_ids": empty_jagged(n, np.int32),
        "beamline_pmt_tdc_ids": empty_jagged(n, np.int32),

        "hit_pmt_calibrated_times": series_to_ak(df["hit_t"], np.float64),
        "hit_pmt_has_time_constant": constant_like(df["hit_t"], 1, np.int8),
    }


# ------------------------------------------------------------
# Build one TTrueInfo chunk
# ------------------------------------------------------------
def build_ttrue_tree(df):
    event_number = df["window_id"].to_numpy(dtype=np.int32)

    return {
        "event_number": event_number,
        "n_overlay": df["n_overlay"].to_numpy(dtype=np.int32),

        "x": series_to_ak(df["x"], np.float32),
        "y": series_to_ak(df["y"], np.float32),
        "z": series_to_ak(df["z"], np.float32),

        "dx": series_to_ak(df["dx"], np.float32),
        "dy": series_to_ak(df["dy"], np.float32),
        "dz": series_to_ak(df["dz"], np.float32),

        "hit_from_prompt": series_to_ak(df["hit_from_prompt"], np.int32),
        "hit_from_capture": series_to_ak(df["hit_from_capture"], np.int32),
        "source_event_idx": series_to_ak(df["source_event_idx"], np.int32),

        "capture_t": series_to_ak(df["capture_t"], np.float64),
        "relative_capture_t": series_to_ak(df["relative_capture_t"], np.float64),
        "prompt_time_withcapture": series_to_ak(df["prompt_time_withcapture"], np.float64),
        "prompt_time": series_to_ak(df["prompt_time"], np.float64),

        "capture_x": series_to_ak(df["capture_x"], np.float32),
        "capture_y": series_to_ak(df["capture_y"], np.float32),
        "capture_z": series_to_ak(df["capture_z"], np.float32),

        "capture_nucleus": series_to_ak(df["capture_nucleus"], np.int32),
        "capture_ngamma": series_to_ak(df["capture_ngamma"], np.int32),
        "capture_total_gammaE": series_to_ak(df["capture_total_gammaE"], np.float32),
    }


# ------------------------------------------------------------
# Write ROOT file with progress bar
# ------------------------------------------------------------
with uproot.recreate(output_root) as f:

    first_chunk = True

    for start in tqdm(range(0, n_windows, chunk_size), desc="Writing ROOT file", unit="chunk"):
        stop = min(start + chunk_size, n_windows)

        df_chunk = windows_df.iloc[start:stop]

        wcte_chunk = build_wcte_tree(df_chunk)
        ttrue_chunk = build_ttrue_tree(df_chunk)

        if first_chunk:
            f["WCTEReadoutWindows"] = wcte_chunk
            f["TTrueInfo"] = ttrue_chunk
            first_chunk = False
        else:
            f["WCTEReadoutWindows"].extend(wcte_chunk)
            f["TTrueInfo"].extend(ttrue_chunk)

        del df_chunk, wcte_chunk, ttrue_chunk
        gc.collect()

print(f"Wrote: {output_root.resolve()}")
print(f"Number of windows written: {n_windows}")

In [ ]:
with uproot.open(output_root) as f:
    print("Top-level keys:")
    for k in f.keys():
        print("  ", k)

    for tree_name in ["WCTEReadoutWindows", "TTrueInfo"]:
        tree = f[tree_name]
        print(f"\nTree: {tree_name}")
        print("Entries:", tree.num_entries)
        print("Branches:")
        for b in tree.keys():
            print(" ", b)

In [ ]:
# TEST: read back the branches from the .root file and inspect them

tree_name = "WCTEReadoutWindows"

with uproot.open(output_root) as f:
    tree = f[tree_name]
    branch_names = tree.keys()
    all_branches = tree.arrays(branch_names, library="ak")

In [ ]:
print("hit_mpmt_slot_ids: ", all_branches["hit_mpmt_slot_ids"][0])
print("hit_mpmt_card_ids: ", all_branches["hit_mpmt_card_ids"][0])
print("hit_pmt_position_ids: ", all_branches["hit_pmt_position_ids"][0])
print("hit_pmt_channel_ids: ", all_branches["hit_pmt_channel_ids"][0])


print("len(hit_pmt_times): ", len(all_branches["hit_pmt_times"][0]))
print("hit_calibrated_times: ", all_branches["hit_pmt_calibrated_times"][0])

---
---
---
---
---

## Efficiency calculation of the DAQ

How many of the neutron captures are lost because they fell outside the readout windows?

In [ ]:
n_windows_total = len(windows_df)

# Total number of events
n_events = windows_df["n_overlay"].sum()

# Total number of captures before windowing, from the original events
n_captures_initial = np.sum(arr["ncaptures"][:n_events])

# Number of windows with at least one recorded capture
n_windows_with_capture = np.sum(windows_df["capture_t"].apply(len) > 0)

# Total number of recorded captures across all windows
n_captures_windowed = np.sum(windows_df["capture_t"].apply(len))

# Ratio
ratio_captures_to_events = n_captures_windowed / n_events
ratio_windowed_to_initial = n_captures_windowed / n_captures_initial

print(f"n_events: {n_events}")

print(f"Total number of windows: {n_windows_total}")
print(f"Windows with at least one recorded capture: {n_windows_with_capture}")
print(f"Total captures before windowing: {n_captures_initial}")
print(f"Total recorded captures in all windows: {n_captures_windowed}")
print()
print(f"Ratio = total captures / total events = {ratio_captures_to_events:.6f}")
print(f"Ratio = total captures / total captures before windowing (efficiency of the DAQ) = {ratio_windowed_to_initial:.6f}")

## Mean and fit of the capture time

In [ ]:
from scipy.optimize import curve_fit

# --- Prepare data ---
t = np.asarray(all_relative_capture_t, dtype=float)
t = t[np.isfinite(t)]
t = t[t >= 0]   # keep only physical capture times

# --- Summary statistics ---
mean_t = t.mean()
print(f"Mean relative capture time = {mean_t:.3f}")

# --- Histogram ---
nbins = 100
counts, bin_edges = np.histogram(t, bins=nbins)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# --- Exponential fit function ---
def expo_model(x, A, tau):
    return A * np.exp(-x / tau)

# Initial guesses
A0 = counts.max()
tau0 = mean_t if mean_t > 0 else 1.0

# Fit only nonzero bins, starting from the second bin
mask = (counts > 0)
mask[0] = False

popt, pcov = curve_fit(
    expo_model,
    bin_centers[mask],
    counts[mask],
    p0=[A0, tau0]
)

A_fit, tau_fit = popt
tau_err = np.sqrt(np.diag(pcov))[1]

print(f"Fitted tau = {tau_fit:.3f} ± {tau_err:.3f}")

# --- Smooth curve for plotting ---
xfit = np.linspace(bin_edges[1], bin_edges[-1], 500)
yfit = expo_model(xfit, A_fit, tau_fit)

# --- Plot ---
plt.figure(figsize=(8, 6))
plt.hist(
    t,
    bins=nbins,
    color="teal",
    alpha=0.75,
    edgecolor="black",
    linewidth=0.8,
    label="Relative capture times"
)
plt.plot(
    xfit,
    yfit,
    color="black",
    linewidth=2.2,
    label=(
        fr"Exponential fit: $\tau = {tau_fit:.2f} \pm {tau_err:.2f}$ ns"
        "\n"
        fr"Mean = {mean_t:.2f} ns"
    )
)

plt.xlabel("Relative capture time [ns]", fontsize=16)
plt.ylabel("Counts", fontsize=16)
plt.title("Distribution of relative capture times", fontsize=18)
plt.tick_params(axis="both", labelsize=13)
plt.legend(fontsize=12)
plt.tight_layout()
plt.show()

## tRMS and Nhits distributions of prompt and captures

In [ ]:
prompt_nhits = []
prompt_trms = []

capture_nhits = []
capture_trms = []

for i in range(len(arr["event"])):

    times = np.asarray(arr["t"][i])

    # ---------------------------
    # Prompt hits in this event
    # ---------------------------
    if arr["has_prompt"][i] != 0:
        prompt_mask = np.asarray(arr["hit_from_prompt"][i]) == 1
        prompt_times = times[prompt_mask]

        if len(prompt_times) > 0 and np.std(prompt_times) > 0: # np.std(prompt_times) > 100
            prompt_nhits.append(len(prompt_times))
            prompt_trms.append(np.std(prompt_times))

    # ---------------------------
    # Capture hits in this event
    # ---------------------------
    if arr["ncaptures"][i] > 0:
        capture_mask = np.asarray(arr["hit_from_capture"][i]) != 0
        capture_times = times[capture_mask]

        if len(capture_times) > 0 and np.std(capture_times) < 100:
            capture_nhits.append(len(capture_times))
            capture_trms.append(np.std(capture_times))


prompt_nhits = np.asarray(prompt_nhits)
prompt_trms = np.asarray(prompt_trms)

capture_nhits = np.asarray(capture_nhits)
capture_trms = np.asarray(capture_trms)

print("Prompt entries:", len(prompt_nhits))
print("Capture entries:", len(capture_nhits))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].hist(prompt_nhits, bins=np.linspace(0,1000,101), color="tab:blue", histtype="step", linewidth=2)
axes[0].set_xlabel("N prompt hits", fontsize=14)
axes[0].set_ylabel("Counts", fontsize=14)
#axes[0].set_yscale("log")
axes[0].set_title("N hits per prompt", fontsize=16)
axes[0].set_xlim(0, 1000)

axes[1].hist(prompt_trms, bins=np.linspace(0,500,300), color="tab:blue", histtype="step", linewidth=2)
axes[1].set_xlabel("Prompt hit time RMS [ns]", fontsize=14)
axes[1].set_ylabel("Counts", fontsize=14)
#axes[1].set_yscale("log")
axes[1].set_title("tRMS per prompt", fontsize=16)
#axes[1].set_xlim(100, 500)
axes[1].set_xlim(0, 500)


hb = axes[2].hexbin(prompt_nhits, prompt_trms, gridsize=80, bins="log", mincnt=1)
axes[2].set_xlabel("N prompt hits", fontsize=14)
axes[2].set_ylabel("Prompt hit time RMS [ns]", fontsize=14)
axes[2].set_title("N prompt hits vs tRMS", fontsize=16)
axes[2].set_xlim(-5, 2500)
axes[2].set_ylim(-5, 1000)

cbar = fig.colorbar(hb, ax=axes[2])
cbar.set_label("Counts")

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].hist(capture_nhits, bins=np.linspace(0,100,51), color="tab:orange", histtype="step", linewidth=2)
axes[0].set_xlabel("N capture hits", fontsize=14)
axes[0].set_ylabel("Counts", fontsize=14)
#axes[0].set_yscale("log")
axes[0].set_title("N hits per capture-origin event", fontsize=16)
axes[0].set_xlim(0, 100)

axes[1].hist(capture_trms, bins=np.linspace(0,12,51), color="tab:orange", histtype="step", linewidth=2)
axes[1].set_xlabel("Capture hit time RMS [ns]", fontsize=14)
axes[1].set_ylabel("Counts", fontsize=14)
#axes[1].set_yscale("log")
axes[1].set_title("tRMS per capture-origin event", fontsize=16)
axes[1].set_xlim(0, 12.5)

hb = axes[2].hexbin(capture_nhits, capture_trms, gridsize=80, bins="log", mincnt=1)
axes[2].set_xlabel("N capture hits", fontsize=14)
axes[2].set_ylabel("Capture hit time RMS [ns]", fontsize=14)
axes[2].set_title("N capture hits vs tRMS", fontsize=16)
axes[2].set_xlim(0, 150)
axes[2].set_ylim(-5, 35)

cbar = fig.colorbar(hb, ax=axes[2])
cbar.set_label("Counts")

plt.tight_layout()
plt.show()

---
---
---

# DEBUG

In [ ]:
# Some prints
idx = 11
print("event =", arr["event"][idx], "subevent =", arr["subevent"][idx])
print("nhits =", arr["ndigi"][idx], "ncaptures =", arr["ncaptures"][idx])
print("capture_t [ns] =", arr["capture_t"][idx])

print("True-hit times [ns] =", arr["t"][idx])
print("hit_from_prompt =", arr["hit_from_prompt"][idx])
print("hit_from_capture =", arr["hit_from_capture"][idx])
print("hit_from_otherprocess =", arr["hit_from_otherprocess"][idx])

print("prompt description:")
print("  has_prompt =", arr["has_prompt"][idx])
print("  prompt_track_id =", arr["prompt_track_id"][idx])
print("  prompt_track_pdg =", arr["prompt_track_pdg"][idx])
print("  prompt_track_energy [MeV] =", arr["prompt_track_energy"][idx])
print("  prompt_track_time [ns] =", arr["prompt_track_time"][idx])
print("  prompt_track_creator_process =", arr["prompt_track_creator_process"][idx])

In [ ]:
# Locate one specific event in the library from the window it belongs to, and print its info
n_windows = 5000

# Sum of the events in the first n windows
n_events= np.sum(windows_df['n_overlay'][:n_windows])
print(f"Sum of n_overlay in the first {n_windows} windows = {n_events}")

# Index of the desired event inside the window
idx_w = 16

#Index of the desired event inside the library
idx = n_events + idx_w

print("event =", arr["event"][idx], "subevent =", arr["subevent"][idx])
print("ndigi =", arr["ndigi"][idx], "ncaptures =", arr["ncaptures"][idx])
print("Hit times [ns] =", arr["t"][idx])
print("capture_t [ns] =", arr["capture_t"][idx])

### Creation processes
| Index | Process                            |
| ----: | ---------------------------------- |
|     0 | `kUnknownProcess`                  |
|     1 | `kDarkNoise`                       |
|     2 | `kHadElastic`                      |
|     3 | `kNCapture`                        |
|     4 | `kCompt`                           |
|     5 | `kPhot`                            |
|     6 | `kCCerenkov`                       |
|     7 | `kSScintillation`                  |
|     8 | `kEBrem`                           |
|     9 | `kAnnihil`                         |
|    10 | `kConv`                            |
|    11 | `kEIoni`                           |
|    12 | `kNeutronInelastic`                |
|    13 | `kMuMinusCaptureAtRest`            |
|    14 | `kMuIoni`                          |
|    15 | `kHIoni`                           |
|    16 | `kDecay`                           |
|    17 | `kRadioactiveDecay`                |
|    18 | `kPhotonNuclear`                   |
|    19 | `kElectronNuclear`                 |
|    20 | `kOpticalPhotonWavelengthShifting` |
|    21 | `kHBertiniCaptureAtRest`           |
|    22 | `kProtonInelastic`                 |
|    23 | `kMuonMinusAtomicCapture`          |
|    24 | `kHadronStoppingProcess`           |
|   ... | ...                                |
|    85 | `kOpMieHG`                         |
|    86 | `kOpAbsorption`                    |
|    87 | `kOpRayleigh`                      |
|    88 | `kOpRaman`                         |
|    89 | `kOpBoundaryProcess`               |
|   ... | ...                                |
|   117 | `kInitialParticle`                 |


---